# Income, Dividends & Projections

The cash the portfolio *throws off* — dividends and interest — and a Monte-Carlo look at
where it might go from here. The income side is actuals from the ledger; the projection is
a simulation, not a forecast.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} across {positions['account_name'].nunique()} accounts "
      f"· {len(history)} trading days {history.index.min():%Y-%m-%d}→{history.index.max():%Y-%m-%d}")

## Dividend calendar

Dividends received by month and year. Bright cells are heavy payout months — the lumpy
quarterly cadence of most payers shows up immediately.

In [ ]:
dmy = A.dividend_month_year(transactions)
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig = px.imshow(dmy.values, x=months, y=[str(y) for y in dmy.index], text_auto=".0f",
                color_continuous_scale="Greens", aspect="auto",
                title=f"Dividend calendar — ${dmy.values.sum():,.0f} total")
fig.update_traces(hovertemplate="%{y} %{x}: $%{z:,.0f}<extra></extra>")
fig.update_layout(height=380, coloraxis_colorbar_title="$")
fig.show()

## Income by year — dividends + interest

Stacked annual income. Interest swelled with cash yields; dividends scale with the equity
and BDC sleeves.

In [ ]:
inc = A.income_by_period(transactions, freq="Y")
inc.index = inc.index.astype(str)
cols = [c for c in ("dividend", "interest") if c in inc.columns]
fig = go.Figure()
for i, c in enumerate(cols):
    fig.add_bar(x=inc.index, y=inc[c], name=c.title(), marker_color=PALETTE[i])
fig.update_layout(barmode="stack", height=420, yaxis_tickprefix="$",
                  title="Annual income", xaxis_title="", hovermode="x unified")
fig.show()
display(inc.style.format("${:,.0f}"))

## Top payers & estimated forward yield

Trailing-12-month income per holding, and that income as a yield on the position's current
value. Money-market sweeps (cash) show income but no yield-on-value here — they're the cash
line, not a security.

In [ ]:
ty = A.trailing_yield(transactions, positions)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Trailing-12m income", "Yield on value"))
top = ty.head(12)
fig.add_bar(y=top.index, x=top["ttm_income"], orientation="h", marker_color=PALETTE[0],
            name="income", row=1, col=1)
yld = ty.dropna(subset=["yield_pct"]).sort_values("yield_pct").tail(12)
fig.add_bar(y=yld.index, x=yld["yield_pct"], orientation="h", marker_color=PALETTE[1],
            name="yield", row=1, col=2)
fig.update_xaxes(tickprefix="$", row=1, col=1)
fig.update_xaxes(ticksuffix="%", row=1, col=2)
fig.update_layout(height=460, showlegend=False, title="Income contribution by holding")
fig.show()

## Monte Carlo — where could this go?

Resample your own daily return distribution forward (block-free bootstrap, keeps the fat
tails) over **5 years**, 2,000 times. The shaded fan is the 5th–95th percentile band; the
line is the median path. This is a distribution of outcomes given history *repeats its
character* — not a prediction.

In [ ]:
HORIZON_Y, N_SIMS = 5, 2000
port_ret = A.portfolio_return_series(positions, history)
paths = A.monte_carlo_paths(port_ret, value0=TOTAL, horizon_days=A.TRADING_DAYS * HORIZON_Y,
                            n_sims=N_SIMS, method="bootstrap")
bands = A.projection_bands(paths)
future = pd.bdate_range(history.index.max(), periods=len(paths))

fig = go.Figure()
fig.add_scatter(x=future, y=bands["p95"], line=dict(width=0), showlegend=False, hoverinfo="skip")
fig.add_scatter(x=future, y=bands["p5"], fill="tonexty", fillcolor="rgba(46,134,171,0.15)",
                line=dict(width=0), name="5–95%")
fig.add_scatter(x=future, y=bands["p75"], line=dict(width=0), showlegend=False, hoverinfo="skip")
fig.add_scatter(x=future, y=bands["p25"], fill="tonexty", fillcolor="rgba(46,134,171,0.30)",
                line=dict(width=0), name="25–75%")
fig.add_scatter(x=future, y=bands["p50"], line=dict(color="#111", width=3), name="median")
fig.add_hline(y=TOTAL, line_dash="dot", line_color="grey", annotation_text="today")
fig.update_layout(height=520, yaxis_tickprefix="$", hovermode="x unified",
                  title=f"{HORIZON_Y}-year Monte Carlo · {N_SIMS:,} bootstrap paths")
fig.show()

final = paths.iloc[-1]
p_gain = (final > TOTAL).mean()
cagr = (bands["p50"].iloc[-1] / TOTAL) ** (1 / HORIZON_Y) - 1
print(f"Median {HORIZON_Y}y outcome: ${bands['p50'].iloc[-1]:,.0f}  (≈{cagr:.1%} CAGR)")
print(f"Range:  p5 ${bands['p5'].iloc[-1]:,.0f}   →   p95 ${bands['p95'].iloc[-1]:,.0f}")
print(f"Probability of ending above today's ${TOTAL:,.0f}: {p_gain:.0%}")

## Distribution of 5-year outcomes

The same simulation as a histogram of ending values — the spread is the real message. The
median, your starting value, and the 5th/95th percentiles are marked.

In [ ]:
fig = px.histogram(final, nbins=60, color_discrete_sequence=[PALETTE[0]],
                   title=f"Distribution of {HORIZON_Y}-year ending values ({N_SIMS:,} sims)")
for v, lbl, color in [(TOTAL, "today", "grey"), (bands["p50"].iloc[-1], "median", "#111"),
                      (bands["p5"].iloc[-1], "p5", "#c0392b"), (bands["p95"].iloc[-1], "p95", "#27ae60")]:
    fig.add_vline(x=v, line_dash="dot", line_color=color, annotation_text=lbl)
fig.update_layout(height=420, xaxis_tickprefix="$", showlegend=False,
                  xaxis_title="ending portfolio value", yaxis_title="simulations")
fig.show()

---
*Income is actuals from the curated ledger. The projection bootstraps historical daily
returns of the **current** composition: it inherits today's concentration, assumes no
future contributions, and — like all such models — understates true tail risk. Directional
intuition, not financial advice.*